# Flower Classifier with Transfer Learning

Classify 5 types of flowers (daisy, dandelion, roses, sunflowers, tulips) using a pretrained **MobileNetV2** backbone and two-phase training.

| Requirement | Section |
|---|---|
| Multi-class dataset, at least 3 classes and 500 images | 1. 5 classes, 3,670 images |
| At least 3 types of data augmentation | 3. Flip, rotation, zoom, contrast |
| Pretrained backbone | 4. MobileNetV2 (ImageNet weights) |
| Two-phase training | 5. Head only, 6. Fine-tune top 30 layers |
| val_accuracy for both phases and final test accuracy | 6. and 7. |
| Confusion matrix | 8. |
| At least 3 sample predictions with confidence | 9. |

## 1. Load the dataset

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

keras.utils.set_random_seed(SEED)

# TensorFlow's flowers dataset: one folder per class
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
downloaded = pathlib.Path(keras.utils.get_file("flower_photos", origin=dataset_url, untar=True))

# The extracted folder location differs between Keras versions, so find the folder that holds the class folders
data_dir = next(downloaded.parent.rglob("daisy")).parent

image_count = len(list(data_dir.glob("*/*.jpg")))
print("Dataset folder:", data_dir)
print("Total images:", image_count)

## 2. Split into training, validation and test sets

80% training, 10% validation, 10% test. The test set is only used once, at the end.

In [ ]:
# subset="both" returns training and validation sets from the same shuffled file list, so they never overlap
train_ds, holdout_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="both",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
)

class_names = train_ds.class_names
num_classes = len(class_names)

# Split the 20% holdout in half: one half for validation, one half for testing
holdout_batches = holdout_ds.cardinality().numpy()
val_ds = holdout_ds.take(holdout_batches // 2)
test_ds = holdout_ds.skip(holdout_batches // 2)

print("Classes:", class_names)
print("Training batches:", train_ds.cardinality().numpy())
print("Validation batches:", val_ds.cardinality().numpy())
print("Test batches:", test_ds.cardinality().numpy())

In [ ]:
# Preview training images to check the labels are correct
images, labels = next(iter(train_ds))

plt.figure(figsize=(12, 6))

for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# cache() keeps decoded images in memory after the first epoch; prefetch() prepares the next batch during training
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

## 3. Data augmentation

Four types of augmentation. They are part of the model, before `preprocess_input`, so they only run during training.

In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.15),
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomContrast(0.1),
], name="data_augmentation")

# Show one training image with 8 different random augmentations
sample_image = images[0]

plt.figure(figsize=(12, 6))

for i in range(8):
    augmented = data_augmentation(sample_image[tf.newaxis, ...], training=True)
    plt.subplot(2, 4, i + 1)
    plt.imshow(augmented[0].numpy().astype("uint8"))
    plt.title(f"Augmented {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Build the model with a frozen MobileNetV2 backbone

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights="imagenet",
)

# Phase 1: keep the pretrained weights unchanged
base_model.trainable = False

inputs = keras.Input(shape=(*IMAGE_SIZE, 3))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(256, activation="relu")(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

trainable_params = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
print(f"Total params:     {model.count_params():,}")
print(f"Trainable params: {trainable_params:,}")

## 5. Phase 1: train the classification head only

In [ ]:
phase1_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=phase1_callbacks,
)

# Evaluate the model that was kept at the end of Phase 1
phase1_val_loss, phase1_val_accuracy = model.evaluate(val_ds, verbose=0)
print(f"\nPhase 1 val_accuracy: {phase1_val_accuracy:.2%}")

## 6. Phase 2: fine-tune the top 30 layers of MobileNetV2

In [ ]:
base_model.trainable = True

# Re-freeze everything except the top 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep Batch Normalization layers frozen for stable fine-tuning
for layer in base_model.layers[-30:]:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

# Recompile with a very small learning rate so the pretrained weights only change slightly
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

trainable_params = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
print(f"Trainable params in Phase 2: {trainable_params:,}")

phase2_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]

# initial_epoch continues the epoch count from where Phase 1 stopped
phase1_epochs = len(history1.history["loss"])

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=phase1_epochs + 20,
    initial_epoch=phase1_epochs,
    callbacks=phase2_callbacks,
)

phase2_val_loss, phase2_val_accuracy = model.evaluate(val_ds, verbose=0)

print(f"\nPhase 1 val_accuracy (head only):  {phase1_val_accuracy:.2%}")
print(f"Phase 2 val_accuracy (fine-tuned): {phase2_val_accuracy:.2%}")
print(f"Change from fine-tuning: {phase2_val_accuracy - phase1_val_accuracy:+.2%}")

In [ ]:
# Validation accuracy across both phases
val_accuracy = history1.history["val_accuracy"] + history2.history["val_accuracy"]
train_accuracy = history1.history["accuracy"] + history2.history["accuracy"]

plt.figure(figsize=(9, 5))
plt.plot(train_accuracy, label="Training accuracy")
plt.plot(val_accuracy, label="Validation accuracy")
plt.axvline(phase1_epochs - 0.5, linestyle="--", color="gray", label="Start of fine-tuning")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Across Both Training Phases")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Final test accuracy

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print("\nResults summary")
print(f"Phase 1 val_accuracy: {phase1_val_accuracy:.2%}")
print(f"Phase 2 val_accuracy: {phase2_val_accuracy:.2%}")
print(f"Final test accuracy:  {test_accuracy:.2%}")

## 8. Classification report and confusion matrix (test set)

In [ ]:
# Collect labels and predictions batch by batch so they stay matched
y_true, y_pred = [], []

for batch_images, batch_labels in test_ds:
    probabilities = model.predict(batch_images, verbose=0)
    y_true.extend(batch_labels.numpy())
    y_pred.extend(probabilities.argmax(axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues", ax=ax, xticks_rotation=45)
plt.title("Confusion Matrix - Test Set")
plt.tight_layout()
plt.show()

## 9. Sample predictions with confidence scores

In [ ]:
sample_images, sample_labels = next(iter(test_ds))
sample_probabilities = model.predict(sample_images[:6], verbose=0)

plt.figure(figsize=(15, 6))

for i in range(6):
    predicted_index = sample_probabilities[i].argmax()
    confidence = sample_probabilities[i][predicted_index]
    actual_index = sample_labels[i].numpy()

    print(
        f"Image {i + 1}: predicted {class_names[predicted_index]} "
        f"({confidence:.1%} confidence), actual {class_names[actual_index]}"
    )

    plt.subplot(2, 3, i + 1)
    plt.imshow(sample_images[i].numpy().astype("uint8"))
    plt.title(
        f"Predicted: {class_names[predicted_index]}\n"
        f"Confidence: {confidence:.1%}\n"
        f"Actual: {class_names[actual_index]}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()